# 01 — Data Ingestion

**IBM Bob assisted** — ingestion pipeline generated via IBM Bob Phase 2 prompt.

This notebook fetches raw NASA DONKI and NOAA SWPC data and serializes it to `data/raw/`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path
import pandas as pd
import json
from datetime import datetime, timedelta, timezone
from dashboard.components.ingestion import ingest_all, fetch_donki, fetch_kp_index, fetch_solar_flux, sha256_file, RAW_DIR

# Configuration
API_KEY = os.environ.get('NASA_API_KEY', 'DEMO_KEY')
END_DATE   = datetime.now(timezone.utc).strftime('%Y-%m-%d')
START_DATE = (datetime.now(timezone.utc) - timedelta(days=365)).strftime('%Y-%m-%d')
print(f'Ingesting {START_DATE} → {END_DATE}')

In [ ]:
results = ingest_all(START_DATE, END_DATE, api_key=API_KEY)
for k, p in results.items():
    print(f'  {k:8s}  {p.name}  sha256={sha256_file(p)[:16]}…')

In [ ]:
# Parse FLR data as a sample
flr_path = RAW_DIR / f'donki_flr_{START_DATE}_{END_DATE}.json'
with open(flr_path) as f:
    flr_raw = json.load(f)
print(f'Solar flare events: {len(flr_raw)}')
if flr_raw:
    flr_df = pd.DataFrame(flr_raw)
    print(flr_df.columns.tolist())
    display(flr_df.head())

In [ ]:
# Parse CME data
cme_path = RAW_DIR / f'donki_cme_{START_DATE}_{END_DATE}.json'
with open(cme_path) as f:
    cme_raw = json.load(f)
print(f'CME events: {len(cme_raw)}')
if cme_raw:
    cme_df = pd.DataFrame(cme_raw)
    display(cme_df.head())

In [ ]:
# Parse GST data
gst_path = RAW_DIR / f'donki_gst_{START_DATE}_{END_DATE}.json'
with open(gst_path) as f:
    gst_raw = json.load(f)
print(f'Geomagnetic storm events: {len(gst_raw)}')
if gst_raw:
    gst_df = pd.DataFrame(gst_raw)
    display(gst_df.head())

In [ ]:
# Parse Kp index
with open(RAW_DIR / 'swpc_kp_1m.json') as f:
    kp_raw = json.load(f)
kp_df = pd.DataFrame(kp_raw)
print(kp_df.dtypes)
display(kp_df.tail())

In [ ]:
# Parse F10.7 solar flux
with open(RAW_DIR / 'swpc_solar_cycle_indices.json') as f:
    flux_raw = json.load(f)
flux_df = pd.DataFrame(flux_raw)
display(flux_df.tail())
print('Columns:', flux_df.columns.tolist())

## Generate Synthetic Launch History (demo scaffold)
Real launch records are merged in `02_eda_and_cleaning.ipynb`.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='7D')
launches = pd.DataFrame({
    'launch_date': date_range[:52],
    'launch_window_utc': [f'{rng.integers(0,24):02d}:00' for _ in range(52)],
    'launch_go': rng.choice([0, 1], size=52, p=[0.3, 0.7]),
})
launches.to_csv(RAW_DIR / 'launch_history.csv', index=False)
print(f'Saved launch_history.csv: {len(launches)} records')
print(launches['launch_go'].value_counts())